In [10]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import mean_squared_error
import warnings
warnings.filterwarnings('ignore')

In [31]:
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import time

class CollaborativeFiltering:
    def __init__(self):
        self.user_item_matrix = None
        self.movies = None
        # --- For Matrix Factorization ---
        self.user_factors = None
        self.item_factors = None
        self.svd_trained = False
        self.als_trained = False

    def load_data(self, path='ml-100k/'):
        """Load MovieLens 100k dataset"""
        try:
            # Load ratings
            ratings = pd.read_csv(f'{path}u.data', sep='\t',
                                  names=['user_id', 'item_id', 'rating', 'timestamp'])

            # Load movies
            self.movies = pd.read_csv(f'{path}u.item', sep='|', encoding='latin-1',
                                      names=['item_id', 'title'] + [f'col_{i}' for i in range(22)])
            self.movies = self.movies[['item_id', 'title']]

            # Create user-item matrix
            self.user_item_matrix = ratings.pivot_table(index='user_id', columns='item_id',
                                                        values='rating', fill_value=0)

            print(f"Loaded {len(ratings)} ratings")
            print(f"Matrix shape: {self.user_item_matrix.shape}")

            return ratings
        except FileNotFoundError:
            print(f"Error: Dataset files not found in the '{path}' directory.")
            print("Please download the MovieLens 100k dataset and place it in the correct folder.")
            return None

    # --- Memory-Based Methods ---

    def user_based_predict(self, user_id, item_id, k=20):
        """Predict rating using User-Based Collaborative Filtering"""
        # ... (existing code from your snippet)
        user_ratings = self.user_item_matrix.loc[user_id].values.reshape(1, -1)
        similarities = cosine_similarity(user_ratings, self.user_item_matrix.values)[0]
        item_ratings = self.user_item_matrix[item_id]
        users_who_rated = item_ratings[item_ratings > 0]
        if len(users_who_rated) == 0:
            return 3.0
        user_sims = []
        user_ratings_list = []
        for user in users_who_rated.index:
            if user != user_id:
                user_idx = self.user_item_matrix.index.get_loc(user)
                sim = similarities[user_idx]
                if sim > 0:
                    user_sims.append(sim)
                    user_ratings_list.append(users_who_rated[user])
        if len(user_sims) == 0:
            return 3.0
        if len(user_sims) > k:
            top_indices = np.argsort(user_sims)[-k:]
            user_sims = [user_sims[i] for i in top_indices]
            user_ratings_list = [user_ratings_list[i] for i in top_indices]
        prediction = np.sum(np.array(user_sims) * np.array(user_ratings_list)) / np.sum(user_sims)
        return prediction

    def item_based_predict(self, user_id, item_id, k=20):
        """Predict rating using Item-Based Collaborative Filtering"""
        # ... (existing code from your snippet)
        item_ratings = self.user_item_matrix[item_id].values.reshape(1, -1)
        similarities = cosine_similarity(item_ratings, self.user_item_matrix.T.values)[0]
        user_ratings = self.user_item_matrix.loc[user_id]
        items_rated = user_ratings[user_ratings > 0]
        if len(items_rated) == 0:
            return 3.0
        item_sims = []
        item_ratings_list = []
        for item in items_rated.index:
            if item != item_id:
                item_idx = self.user_item_matrix.columns.get_loc(item)
                sim = similarities[item_idx]
                if sim > 0:
                    item_sims.append(sim)
                    item_ratings_list.append(items_rated[item])
        if len(item_sims) == 0:
            return 3.0
        if len(item_sims) > k:
            top_indices = np.argsort(item_sims)[-k:]
            item_sims = [item_sims[i] for i in top_indices]
            item_ratings_list = [item_ratings_list[i] for i in top_indices]
        prediction = np.sum(np.array(item_sims) * np.array(item_ratings_list)) / np.sum(item_sims)
        return prediction

    # --- Model-Based Methods (SVD and ALS) ---

    def train_svd(self, n_factors=20, n_epochs=20, lr=0.01, reg=0.02):
        """Train model using Funk SVD with Stochastic Gradient Descent."""
        print("Training SVD model...")
        n_users, n_items = self.user_item_matrix.shape
        self.user_factors = np.random.rand(n_users, n_factors) * 0.1
        self.item_factors = np.random.rand(n_items, n_factors) * 0.1
        
        user_map = {user_id: i for i, user_id in enumerate(self.user_item_matrix.index)}
        item_map = {item_id: i for i, item_id in enumerate(self.user_item_matrix.columns)}

        ratings_arr = self.user_item_matrix.to_numpy()
        
        for epoch in range(n_epochs):
            start_time = time.time()
            for u_idx, i_idx in np.argwhere(ratings_arr > 0):
                rating = ratings_arr[u_idx, i_idx]
                
                # Predict and calculate error
                pred = self.user_factors[u_idx, :] @ self.item_factors[i_idx, :].T
                error = rating - pred
                
                # Update factors
                uf = self.user_factors[u_idx, :]
                itf = self.item_factors[i_idx, :]
                self.user_factors[u_idx, :] += lr * (error * itf - reg * uf)
                self.item_factors[i_idx, :] += lr * (error * uf - reg * itf)
            
            print(f"Epoch {epoch+1}/{n_epochs} completed in {time.time() - start_time:.2f}s")
            
        self.svd_trained = True
        self.als_trained = False # Invalidate ALS if SVD is trained

    def train_als(self, n_factors=20, n_epochs=10, reg=0.1):
        """Train model using Alternating Least Squares."""
        print("Training ALS model...")
        n_users, n_items = self.user_item_matrix.shape
        self.user_factors = np.random.rand(n_users, n_factors) * 0.1
        self.item_factors = np.random.rand(n_items, n_factors) * 0.1
        
        ratings_arr = self.user_item_matrix.to_numpy()
        I = np.eye(n_factors)

        for epoch in range(n_epochs):
            start_time = time.time()
            # Fix item factors, solve for user factors
            for u_idx in range(n_users):
                rated_items_indices = np.where(ratings_arr[u_idx, :] > 0)[0]
                if not len(rated_items_indices): continue
                
                Q_u = self.item_factors[rated_items_indices, :]
                ratings_u = ratings_arr[u_idx, rated_items_indices]
                
                A = Q_u.T @ Q_u + reg * I
                b = Q_u.T @ ratings_u
                self.user_factors[u_idx, :] = np.linalg.solve(A, b)

            # Fix user factors, solve for item factors
            for i_idx in range(n_items):
                rating_users_indices = np.where(ratings_arr[:, i_idx] > 0)[0]
                if not len(rating_users_indices): continue

                P_i = self.user_factors[rating_users_indices, :]
                ratings_i = ratings_arr[rating_users_indices, i_idx]
                
                A = P_i.T @ P_i + reg * I
                b = P_i.T @ ratings_i
                self.item_factors[i_idx, :] = np.linalg.solve(A, b)

            print(f"Epoch {epoch+1}/{n_epochs} completed in {time.time() - start_time:.2f}s")

        self.als_trained = True
        self.svd_trained = False # Invalidate SVD if ALS is trained

    def matrix_factorization_predict(self, user_id, item_id):
        """Predict rating using trained matrix factorization model."""
        try:
            user_idx = self.user_item_matrix.index.get_loc(user_id)
            item_idx = self.user_item_matrix.columns.get_loc(item_id)
            
            prediction = self.user_factors[user_idx, :] @ self.item_factors[item_idx, :].T
            return prediction
        except KeyError:
            return 3.0 # Default rating for unknown user/item

    def get_recommendations(self, user_id, method='user_based', n_recs=5):
        """Get top N recommendations for a user"""
        user_ratings = self.user_item_matrix.loc[user_id]
        unrated_items = user_ratings[user_ratings == 0].index
        
        predictions = []
        print(f"Calculating predictions for user {user_id} using '{method}'...")
        
        # Ensure model is trained if using SVD or ALS
        if method == 'svd' and not self.svd_trained:
            print("SVD model not trained. Training now with default parameters.")
            self.train_svd()
        elif method == 'als' and not self.als_trained:
            print("ALS model not trained. Training now with default parameters.")
            self.train_als()

        # For speed, we sample unrated items. For a real system, you'd predict on more.
        sample_size = min(200, len(unrated_items))
        sample_items = np.random.choice(unrated_items, sample_size, replace=False)
        
        for item_id in sample_items:
            if method == 'user_based':
                pred = self.user_based_predict(user_id, item_id)
            elif method == 'item_based':
                pred = self.item_based_predict(user_id, item_id)
            elif method in ['svd', 'als']:
                pred = self.matrix_factorization_predict(user_id, item_id)
            else:
                raise ValueError("Method must be 'user_based', 'item_based', 'svd', or 'als'")
            predictions.append((item_id, pred))
        
        predictions.sort(key=lambda x: x[1], reverse=True)
        top_recs = predictions[:n_recs]
        
        recommendations = []
        for item_id, rating in top_recs:
            title = self.movies[self.movies['item_id'] == item_id]['title'].iloc[0]
            recommendations.append((title, rating))
            
        return recommendations

    def show_user_profile(self, user_id, n_movies=5):
        # ... (existing code from your snippet)
        user_ratings = self.user_item_matrix.loc[user_id]
        top_rated = user_ratings[user_ratings > 0].sort_values(ascending=False).head(n_movies)
        print(f"\nUser {user_id}'s Top Rated Movies:")
        for item_id, rating in top_rated.items():
            title = self.movies[self.movies['item_id'] == item_id]['title'].iloc[0]
            print(f"  {title}: {rating}")
        return top_rated




In [32]:
# --- Example Usage ---
if __name__ == '__main__':
    cf = CollaborativeFiltering()
    ratings_data = cf.load_data()

    if ratings_data is not None:
        target_user = 30
        cf.show_user_profile(target_user)

        print("\n--- User-Based Recommendations ---")
        recs_user = cf.get_recommendations(target_user, method='user_based')
        for title, rating in recs_user:
            print(f"  {title} (Predicted Rating: {rating:.2f})")

        print("\n--- Item-Based Recommendations ---")
        recs_item = cf.get_recommendations(target_user, method='item_based')
        for title, rating in recs_item:
            print(f"  {title} (Predicted Rating: {rating:.2f})")

        print("\n--- SVD-Based Recommendations ---")
        # Train SVD model first (this might take a minute)
        cf.train_svd(n_factors=30, n_epochs=25, lr=0.005, reg=0.02)
        recs_svd = cf.get_recommendations(target_user, method='svd')
        for title, rating in recs_svd:
            print(f"  {title} (Predicted Rating: {rating:.2f})")
            
        print("\n--- ALS-Based Recommendations ---")
        # Train ALS model first (this might take a minute)
        cf.train_als(n_factors=30, n_epochs=10, reg=0.1)
        recs_als = cf.get_recommendations(target_user, method='als')
        for title, rating in recs_als:
            print(f"  {title} (Predicted Rating: {rating:.2f})")

Loaded 100000 ratings
Matrix shape: (943, 1682)

User 30's Top Rated Movies:
  Contact (1997): 5.0
  Shine (1996): 5.0
  Waiting for Guffman (1996): 5.0
  Butch Cassidy and the Sundance Kid (1969): 5.0
  Forrest Gump (1994): 5.0

--- User-Based Recommendations ---
Calculating predictions for user 30 using 'user_based'...
  Princess Bride, The (1987) (Predicted Rating: 4.54)
  Great Escape, The (1963) (Predicted Rating: 4.44)
  Shall We Dance? (1996) (Predicted Rating: 4.40)
  Wallace & Gromit: The Best of Aardman Animation (1996) (Predicted Rating: 4.35)
  African Queen, The (1951) (Predicted Rating: 4.19)

--- Item-Based Recommendations ---
Calculating predictions for user 30 using 'item_based'...
  Very Natural Thing, A (1974) (Predicted Rating: 5.00)
  Mirage (1995) (Predicted Rating: 4.54)
  Raise the Red Lantern (1991) (Predicted Rating: 4.43)
  Ruby in Paradise (1993) (Predicted Rating: 4.40)
  Mediterraneo (1991) (Predicted Rating: 4.35)

--- SVD-Based Recommendations ---
Traini